In [ ]:
from openai import OpenAI
import json, numpy as np, time, re
import os
import json
import hashlib
import re
import numpy as np
from typing import List, Dict, Tuple
from google.colab import userdata

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
)

MODEL = "qwen/qwen3.5-122b-a10b"


ModuleNotFoundError: No module named 'google.colab'

In [9]:

def chat(messages, model=MODEL, temperature=0.3, max_tokens=1024):
    resp = client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content

print(chat([{"role": "user", "content": "Say 'hello world' and nothing else."}]))

AuthenticationError: Error code: 401 - {'error': {'message': 'User not found.', 'code': 401}}

In [ ]:
# === MULTI-HOP NEEDLE ===
# The model must find TWO facts and connect them:
# Fact 1: "Dr. Elena Vasquez leads the AURORA-7 initiative."
# Fact 2: "Dr. Vasquez confirmed the launch window opens on March 15, 2024."
# Question: "What is the launch date for AURORA-7?"
# The model needs to connect Vasquez -> AURORA-7 -> March 15.

FACT_A = "Dr. Elena Vasquez was appointed as the lead researcher for the classified AURORA-7 initiative after her successful tenure directing the Meridian program."
FACT_B = "In a private briefing last Tuesday, Dr. Vasquez confirmed that the launch window for her current project opens on March 15, 2024, pending final safety reviews."

# Filler paragraphs about similar topics (projects, dates, people)
FILLER_POOL = [
    "The quarterly infrastructure review highlighted upgrades to the east-coast data centers. Migration timelines were set for late Q2. The operations team reported a 99.97 percent uptime for the previous period and proposed additional redundancy measures for the backup systems.",
    "Dr. James Morton presented findings from the computational biology lab. The new protein folding algorithm showed a 34 percent improvement in prediction accuracy. Funding proposals were submitted to three federal agencies for continued research.",
    "Project TITAN-6 entered Phase 3 clinical evaluation under the supervision of Dr. Sarah Lin. Preliminary results are expected by August 2024. The regulatory affairs team has begun preparing submission documents for the oversight board.",
    "The advanced materials team tested a new ceramic composite for thermal shielding. Results exceeded expectations at temperatures above 2000 degrees Celsius. A patent application was filed and manufacturing partners have been contacted.",
    "Chief Financial Officer Mark Reynolds presented the revised budget allocations for fiscal year 2025. Research and development spending will increase by 18 percent. Capital expenditure for new laboratory facilities was approved at 47 million dollars.",
    "The cybersecurity division completed its annual red team exercise across all classified networks. Two medium-severity vulnerabilities were discovered and patched within 72 hours. New endpoint detection protocols were deployed organization-wide.",
    "Professor Lisa Chang published her team's findings on quantum entanglement stability in the latest issue of Physical Review Letters. The results suggest a viable path toward error-corrected quantum computing within the next decade.",
    "Project ORION-11 received continued funding approval from the defense advisory panel. The next milestone review is scheduled for May 2024. Program manager David Kowalski noted that the project remains on schedule and within budget parameters.",
    "The talent acquisition team reported hiring 43 new researchers across six departments. Retention rates improved to 94 percent following the introduction of flexible work arrangements. The mentorship program expanded to include 120 pairs.",
    "Environmental monitoring stations detected a slight increase in seismic activity near the northern test facility. Geological surveys confirmed no structural risk. Additional monitoring equipment has been installed as a precautionary measure.",
    "The satellite communications upgrade was completed two weeks ahead of schedule. Bandwidth capacity doubled to support the growing number of field operations. Dr. Robert Kim oversaw the final integration testing phase and signed off on deployment.",
    "Project NEBULA-4 concluded its preliminary design review. The propulsion subsystem exceeded thrust-to-weight requirements by 12 percent. Systems engineering lead Patricia Gomez recommended advancing to the detailed design phase in Q3.",
    "The medical research wing reported promising results from the Phase 2 trial of compound MRX-7821. Efficacy rates reached 67 percent in the target population. An expanded trial involving 2000 additional participants was approved for the next quarter.",
    "Facilities management completed the renovation of Building 7, adding 15000 square feet of laboratory space. The new cleanroom meets ISO Class 5 standards. Occupancy is planned for early February 2024.",
    "Dr. Amanda Foster's team demonstrated a novel machine learning approach for anomaly detection in network traffic. The system reduced false positive rates by 40 percent compared to existing methods used across the organization.",
    "The supply chain optimization initiative yielded a 22 percent reduction in procurement lead times. Strategic partnerships with three new vendors were established. Inventory carrying costs decreased by 1.3 million dollars annually.",
    "The HELIOS-8 solar energy research program completed its first full year of field testing. Energy conversion efficiency averaged 31.4 percent across all test sites. Results will be presented at the International Energy Conference in Geneva.",
    "Human resources launched the updated professional development framework. Over 300 employees enrolled in the advanced leadership track. The tuition reimbursement budget was increased by 25 percent to accommodate growing demand.",
    "The autonomous systems laboratory tested its latest navigation algorithm in simulated urban environments. The system achieved a 98.7 percent obstacle avoidance rate. Real-world field trials are planned for the second half of 2024.",
    "Project AEGIS-3 was formally closed after achieving all deliverables ahead of schedule. The final report documented 14 technical innovations and 6 patent filings. Lessons learned sessions were conducted with all participating teams."
]

print(f"Fact A: {FACT_A}")
print(f"Fact B: {FACT_B}")
print(f"Filler paragraphs available: {len(FILLER_POOL)}")

Fact A: Dr. Elena Vasquez was appointed as the lead researcher for the classified AURORA-7 initiative after her successful tenure directing the Meridian program.
Fact B: In a private briefing last Tuesday, Dr. Vasquez confirmed that the launch window for her current project opens on March 15, 2024, pending final safety reviews.
Filler paragraphs available: 20


In [10]:
# === CONTEXT ROT EXPERIMENT ===

QUESTION = "What is the launch date for AURORA-7?"
# Correct answer requires: Fact A (Vasquez leads AURORA-7) + Fact B (Vasquez launch = March 15)

def build_multihop_haystack(n_filler):
    """Build a haystack with TWO facts separated by filler."""
    fillers = (FILLER_POOL * ((n_filler // len(FILLER_POOL)) + 1))[:n_filler]
    np.random.seed(42)
    np.random.shuffle(fillers)
    pos_a = max(1, len(fillers) // 4)
    pos_b = (3 * len(fillers)) // 4
    fillers.insert(pos_a, FACT_A)
    fillers.insert(pos_b, FACT_B)
    return "\n\n".join(fillers)

def test_multihop(context, question):
    msg = [
        {"role": "system", "content": "Answer the question based ONLY on the provided context. Be precise and specific."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
    ]
    return chat(msg, max_tokens=200)

experiments = [
    {"name": "Tiny (2 facts + 5 fillers)",    "n_filler": 5},
    {"name": "Small (2 facts + 20 fillers)",   "n_filler": 20},
    {"name": "Medium (2 facts + 60 fillers)",  "n_filler": 60},
    {"name": "Large (2 facts + 120 fillers)",  "n_filler": 120},
    {"name": "XL (2 facts + 200 fillers)",     "n_filler": 200},
]

print("CONTEXT ROT EXPERIMENT (Multi-Hop)")
print("=" * 60)
print(f"Question: {QUESTION}")
print(f"Correct: March 15, 2024 (requires connecting Vasquez -> AURORA-7 -> date)")
print()

for exp in experiments:
    ctx = build_multihop_haystack(exp["n_filler"])
    n_words = len(ctx.split())
    approx_tokens = n_words * 4 // 3
    print(f"\n>> {exp['name']} (~{n_words} words, ~{approx_tokens} tokens)")
    answer = test_multihop(ctx, QUESTION)
    correct = "march 15" in answer.lower()
    status = "[PASS]" if correct else "[FAIL]"
    print(f"   {status} Answer: {answer[:250]}")
    time.sleep(1)

print("\n" + "=" * 60)
print("Multi-hop reasoning degrades much faster than simple lookup.")
print("The model must connect Vasquez -> AURORA-7 -> March 15 date.")
print("As context grows, the attention budget gets stretched thin.")
print("This is context rot.")

CONTEXT ROT EXPERIMENT (Multi-Hop)
Question: What is the launch date for AURORA-7?
Correct: March 15, 2024 (requires connecting Vasquez -> AURORA-7 -> date)



NameError: name 'FILLER_POOL' is not defined

In [11]:
# --- Strategy 1: Fixed-size (the naive way) ---

def chunk_fixed(text: str, chunk_size: int = 200, overlap: int = 50) -> List[str]:
    """Split text into fixed-size word chunks with overlap."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk.strip())
    return chunks

In [12]:
# --- Strategy 2: Recursive/Semantic (split on natural boundaries) ---

def chunk_recursive(text: str, max_size: int = 200) -> List[str]:
    """Split text on natural boundaries: paragraphs, then sentences."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

    chunks = []
    for para in paragraphs:
        words = para.split()
        if len(words) <= max_size:
            chunks.append(para)
        else:
            sentences = para.replace(". ", ".\n").split("\n")
            current, current_len = [], 0
            for sent in sentences:
                sent_len = len(sent.split())
                if current_len + sent_len > max_size and current:
                    chunks.append(" ".join(current))
                    current, current_len = [sent], sent_len
                else:
                    current.append(sent)
                    current_len += sent_len
            if current:
                chunks.append(" ".join(current))

    return [c for c in chunks if len(c.split()) > 10]

In [13]:
# --- Compare them ---

sample = DOCUMENTS[0]["content"]

fixed = chunk_fixed(sample, chunk_size=100, overlap=20)
recursive = chunk_recursive(sample, max_size=100)

print("=" * 60)
print("FIXED-SIZE CHUNKING")
print("=" * 60)
for i, c in enumerate(fixed[:3]):
    print(f"\nChunk {i+1} ({len(c.split())} words):")
    print(c[:150], "...")

print("\n" + "=" * 60)
print("RECURSIVE CHUNKING")
print("=" * 60)
for i, c in enumerate(recursive[:3]):
    print(f"\nChunk {i+1} ({len(c.split())} words):")
    print(c[:150], "...")

print("\n💡 Fixed cuts mid-thought. Recursive respects paragraph boundaries.")

NameError: name 'DOCUMENTS' is not defined

In [14]:
from openai import OpenAI
import chromadb

oai = OpenAI()

def get_embeddings(texts: List[str], model: str = "text-embedding-3-small") -> List[List[float]]:
    """Batch embed texts with OpenAI."""
    cleaned = [t.replace("\n", " ").strip() for t in texts]
    resp = oai.embeddings.create(input=cleaned, model=model)
    return [d.embedding for d in resp.data]

def get_embedding(text: str) -> List[float]:
    return get_embeddings([text])[0]

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [15]:
# --- Chunk all documents ---

all_chunks = []
chunk_meta = []

for doc in DOCUMENTS:
    chunks = chunk_recursive(doc["content"], max_size=100)
    for chunk in chunks:
        all_chunks.append(chunk)
        chunk_meta.append({"title": doc["title"], "source": doc["source"]})

print(f"Total chunks: {len(all_chunks)}")
for doc in DOCUMENTS:
    n = sum(1 for m in chunk_meta if m["title"] == doc["title"])
    print(f"  {doc['title']}: {n} chunks")

NameError: name 'DOCUMENTS' is not defined

In [16]:
# --- Embed and store in ChromaDB ---

chroma = chromadb.Client()

try: chroma.delete_collection("naive_rag")
except: pass

collection = chroma.create_collection("naive_rag", metadata={"hnsw:space": "cosine"})

embs = get_embeddings(all_chunks)

collection.add(
    ids=[f"chunk_{i}" for i in range(len(all_chunks))],
    embeddings=embs,
    documents=all_chunks,
    metadatas=chunk_meta
)

print(f"✅ Stored {len(all_chunks)} chunks in ChromaDB")

NameError: name 'get_embeddings' is not defined

In [17]:
# --- The naive RAG query function ---

def naive_rag(question: str, k: int = 5, verbose: bool = True) -> str:
    """Simplest RAG: semantic search → stuff prompt → generate."""

    results = collection.query(
        query_embeddings=[get_embedding(question)],
        n_results=k
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0]

    if verbose:
        print(f"\n🔍 Query: '{question}'")
        print(f"\nRetrieved {k} chunks:")
        for i, (d, m, dist) in enumerate(zip(docs, metas, dists)):
            print(f"  [{i+1}] dist={dist:.3f} | {m['title']}")
            print(f"      {d[:80]}...")

    context = "\n".join(
        f"[Source: {m['title']}]\n{d}" for d, m in zip(docs, metas)
    )

    resp = oai.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role": "user", "content": f"""Answer based ONLY on the context below.
If the context doesn't have the answer, say \"I don't have enough information.\"
Cite your sources.

Context:
{context}

Question: {question}

Answer:"""}]
    )

    answer = resp.choices[0].message.content
    if verbose:
        print(f"\n💬 Answer:\n{answer}")
    return answer

In [18]:
from rank_bm25 import BM25Okapi

def tokenize(text: str) -> List[str]:
    """Simple whitespace + lowercase tokenizer for BM25."""
    return re.findall(r'\w+', text.lower())

# Build BM25 index over the same chunks
bm25 = BM25Okapi([tokenize(c) for c in all_chunks])

print(f"✅ Built BM25 index over {len(all_chunks)} chunks")

ZeroDivisionError: division by zero

In [19]:
def reciprocal_rank_fusion(
    semantic: List[Tuple[int, float]],
    keyword: List[Tuple[int, float]],
    k: int = 60
) -> List[Tuple[int, float]]:
    """
    Merge two ranked lists with RRF.
    Simple, effective, no hyperparameters to tune.
    """
    scores = {}
    for rank, (idx, _) in enumerate(semantic):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank + 1)
    for rank, (idx, _) in enumerate(keyword):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

In [20]:
def hybrid_search(question: str, k: int = 5) -> List[Dict]:
    """Combine semantic + BM25 with RRF."""

    # Semantic search (broad, top 20)
    sem = collection.query(query_embeddings=[get_embedding(question)], n_results=20)
    sem_ranked = [
        (int(id.split("_")[1]), dist)
        for id, dist in zip(sem["ids"][0], sem["distances"][0])
    ]

    # BM25 keyword search (top 20)
    scores = bm25.get_scores(tokenize(question))
    bm25_ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:20]

    # Fuse
    fused = reciprocal_rank_fusion(sem_ranked, bm25_ranked)

    return [
        {"chunk": all_chunks[idx], "meta": chunk_meta[idx], "score": sc}
        for idx, sc in fused[:k]
    ]

In [21]:
def hybrid_rag(question: str, k: int = 5, verbose: bool = True) -> str:
    """RAG with hybrid search."""
    results = hybrid_search(question, k)

    if verbose:
        print(f"\n🔍 Query: '{question}'")
        print(f"\nRetrieved {len(results)} chunks (hybrid):")
        for i, r in enumerate(results):
            print(f"  [{i+1}] RRF={r['score']:.4f} | {r['meta']['title']}")
            print(f"      {r['chunk'][:80]}...")

    context = "\n".join(
        f"[Source: {r['meta']['title']}]\n{r['chunk']}" for r in results
    )

    resp = oai.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role": "user", "content": f"""Answer based ONLY on the context below.
If the context doesn't have the answer, say \"I don't have enough information.\"
Cite your sources.

Context:
{context}

Question: {question}

Answer:"""}]
    )

    answer = resp.choices[0].message.content
    if verbose:
        print(f"\n💬 Answer:\n{answer}")
    return answer

In [22]:
from anthropic import Anthropic

ant = Anthropic()

def contextualize_chunk(chunk: str, full_doc: str, title: str) -> str:
    """Prepend LLM-generated context to a chunk before embedding."""
    resp = ant.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=150,
        messages=[{"role": "user", "content": f"""<document title="{title}">
{full_doc}
</document>

Here is a chunk from that document:
<chunk>
{chunk}
</chunk>

Write a SHORT (2-3 sentence) context that situates this chunk within the document.
Include: which document, what section/topic, key entities or time periods.
This will be prepended to the chunk for search.

Context:"""}]
    )
    ctx = resp.content[0].text.strip()
    return f"{ctx}\n\n{chunk}"

In [23]:
# --- Contextualize all chunks (makes LLM calls — takes 1-2 min) ---

print("Contextualizing chunks... (LLM call per chunk, patience)")

ctx_chunks = []
for i, (chunk, meta) in enumerate(zip(all_chunks, chunk_meta)):
    doc = next(d for d in DOCUMENTS if d["title"] == meta["title"])
    ctx_chunks.append(contextualize_chunk(chunk, doc["content"], doc["title"]))
    if (i + 1) % 5 == 0:
        print(f"  {i+1}/{len(all_chunks)} done")

print(f"\n✅ Contextualized {len(ctx_chunks)} chunks")

Contextualizing chunks... (LLM call per chunk, patience)

✅ Contextualized 0 chunks


In [24]:
# --- Re-embed and store contextualized chunks ---

try: chroma.delete_collection("ctx_rag")
except: pass

ctx_collection = chroma.create_collection("ctx_rag", metadata={"hnsw:space": "cosine"})

ctx_embs = get_embeddings(ctx_chunks)
ctx_collection.add(
    ids=[f"ctx_{i}" for i in range(len(ctx_chunks))],
    embeddings=ctx_embs,
    documents=ctx_chunks,
    metadatas=chunk_meta
)

# BM25 on contextualized chunks too
ctx_bm25 = BM25Okapi([tokenize(c) for c in ctx_chunks])

print(f"✅ Stored {len(ctx_chunks)} contextualized chunks")

NameError: name 'get_embeddings' is not defined

In [25]:
def ctx_hybrid_search(question: str, k: int = 5) -> List[Dict]:
    """Hybrid search over contextualized chunks."""
    sem = ctx_collection.query(query_embeddings=[get_embedding(question)], n_results=20)
    sem_ranked = [
        (int(id.split("_")[1]), dist)
        for id, dist in zip(sem["ids"][0], sem["distances"][0])
    ]

    scores = ctx_bm25.get_scores(tokenize(question))
    bm25_ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:20]

    fused = reciprocal_rank_fusion(sem_ranked, bm25_ranked)

    return [
        {
            "chunk": ctx_chunks[idx],
            "original": all_chunks[idx],
            "meta": chunk_meta[idx],
            "score": sc
        }
        for idx, sc in fused[:k]
    ]

In [26]:
# --- Test: the cross-doc question that's hard for naive RAG ---

results = ctx_hybrid_search("What is ACME's AI strategy and how does it connect to current products?")
print("Top 3 contextualized results:")
for i, r in enumerate(results[:3]):
    print(f"\n[{i+1}] {r['meta']['title']}")
    print(f"    {r['chunk'][:150]}...")

NameError: name 'get_embedding' is not defined

In [27]:
# --- Test: the cross-doc question that's hard for naive RAG ---

results = ctx_hybrid_search("What is ACME's AI strategy and how does it connect to current products?")
print("Top 3 contextualized results:")
for i, r in enumerate(results[:3]):
    print(f"\n[{i+1}] {r['meta']['title']}")
    print(f"    {r['chunk'][:150]}...")

NameError: name 'get_embedding' is not defined

In [28]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✅ Loaded cross-encoder reranker")

c:\Users\HP\Desktop\codes\ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5494.24it/s]


✅ Loaded cross-encoder reranker


In [29]:
def rerank(question: str, results: List[Dict], top_k: int = 3) -> List[Dict]:
    """Rerank results with cross-encoder."""
    pairs = [(question, r["chunk"]) for r in results]
    scores = reranker.predict(pairs)
    for r, s in zip(results, scores):
        r["rerank_score"] = float(s)
    return sorted(results, key=lambda x: x["rerank_score"], reverse=True)[:top_k]

In [30]:
def full_rag(question: str, verbose: bool = True) -> str:
    """
    The full pipeline:
    Contextual chunks → Hybrid search (top 10) → Rerank (top 3) → Generate
    """
    results = ctx_hybrid_search(question, k=10)
    top = rerank(question, results, top_k=3)

    if verbose:
        print(f"\n🔍 Query: '{question}'")
        print(f"\nTop 3 after reranking:")
        for i, r in enumerate(top):
            print(f"  [{i+1}] rerank={r['rerank_score']:.3f} | {r['meta']['title']}")
            print(f"      {r['chunk'][:100]}...")

    context = "\n".join(
        f"[Source: {r['meta']['title']}]\n{r['chunk']}" for r in top
    )

    resp = oai.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role": "user", "content": f"""Answer based ONLY on the context below.
If the context doesn't have the answer, say \"I don't have enough information.\"
Cite your sources.

Context:
{context}

Question: {question}

Answer:"""}]
    )

    answer = resp.choices[0].message.content
    if verbose:
        print(f"\n💬 Answer:\n{answer}")
    return answer